# Panel Construction

Produces **8 Stata .dta files** consumed by `analysis.R`:

| File | Used for |
|------|----------|
| `firm_quarter_stock_active_ge20_chatgpt_treated_pretrend2021.dta` | Baseline DiD, baseline chars, mechanism plots |
| `firm_quarter_new_hires_active_ge20_chatgpt_treated_pretrend2021.dta` | Baseline DiD, placebo, mechanism plots |
| `firm_quarter_separations_active_ge20_chatgpt_treated_pretrend2021.dta` | Baseline DiD |
| `firm_quarter_promotions_active_ge20_chatgpt_treated_pretrend2021.dta` | Baseline DiD |
| `stock_staggered.dta` | CS + SA staggered DiD |
| `hires_staggered.dta` | CS + SA staggered DiD |
| `seps_staggered.dta` | CS + SA staggered DiD |
| `promos_staggered.dta` | CS + SA staggered DiD |

The stock and hires flat panels include seniority-split and AI-exposure columns merged in,
as these are needed by the mechanism plots in `analysis.R`.

Run all cells top-to-bottom. Edit only the **Config** cell.

## 1. Imports

In [ ]:
import os
import subprocess
import sys

import numpy as np
import pandas as pd
from tqdm import tqdm

try:
    import pyreadstat
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pyreadstat", "-q"])
    import pyreadstat


## 2. Config

In [ ]:
# ── Input paths ───────────────────────────────────────────────────────────────
POSITIONS_PATH    = "../data/Positions/positions_stock.csv"
POST1_PATH        = "../data/Firm Level/flagged_postings_gpt_predictions.csv"
POST2_PATH        = "../data/Firm Level/flagged_postings_gpt_predictions_2025.csv"
OCC_EXPOSURE_PATH = "../data/AI Exposure Scores/occ_level.csv"

# ── Output directory ──────────────────────────────────────────────────────────
OUT_DIR = "../data/Positions/"
os.makedirs(OUT_DIR, exist_ok=True)

# ── Panel parameters ──────────────────────────────────────────────────────────
# NOTE: PANEL_START = 2021Q1 to match the pretrend2021 filenames used by analysis.R
PANEL_START   = "2021Q1"
PANEL_END     = "2025Q2"
TREAT_Q       = "2022Q4"
HIRE_START    = "2021-01-01"
HIRE_END      = "2025-07-01"
TREATED_START = "2022-11-30"
TREATED_END   = "2025-07-01"
THRESHOLD     = 20            # analysis.R only uses ge20

# ── Industry exclusions ───────────────────────────────────────────────────────
RECRUITMENT_INDUSTRIES = [
    "Recruitment and Staffing Services",
    "Employment and Staffing Services",
    "Employment and Recruitment Services",
    "Human Resources and Recruitment Services",
    "Online Employment Platforms",
    "Human Resources and Workforce Solutions",
    "Business Process Outsourcing Services",
]

# ── AI exposure column in occ_level.csv ───────────────────────────────────────
EXPOSURE_VALUE_COL = "human_rating_beta"
EXPOSURE_ONET_COL  = "O*NET-SOC Code"

STATA_EPOCH = pd.Period("1960Q1", freq="Q")


## 3. Shared Utilities

In [ ]:
def make_firm_quarter_grid(active_firms: set, quarters: pd.PeriodIndex) -> pd.DataFrame:
    return (
        pd.DataFrame({"company_name": sorted(active_firms)}).assign(_tmp=1)
        .merge(pd.DataFrame({"quarter": quarters}).assign(_tmp=1), on="_tmp")
        .drop(columns="_tmp")
    )


def add_common_panel_columns(panel: pd.DataFrame, treated_firms: set,
                              treat_q: str) -> pd.DataFrame:
    panel = panel.copy()
    panel["treated"]      = panel["company_name"].isin(treated_firms).astype(int)
    panel["Treatment"]    = panel["treated"].map({0: "Control", 1: "Ever Treated"})
    panel["post"]         = (panel["quarter"] >= pd.Period(treat_q, freq="Q")).astype(int)
    panel["quarter_date"] = panel["quarter"].dt.to_timestamp()
    panel["firm_id"]      = panel["company_name"].astype("category").cat.codes + 1
    panel["tq"]           = panel["quarter"].apply(lambda p: (p - STATA_EPOCH).n).astype(int)
    return panel


def save_dta(df: pd.DataFrame, path: str, int_cols: list, float_cols: list) -> None:
    out = df.copy()
    for c in int_cols:   out[c] = out[c].astype(int)
    for c in float_cols: out[c] = out[c].astype(float)
    pyreadstat.write_dta(out, path, version=14)
    print(f"  Saved → {path}  ({len(out):,} rows)")


def print_firm_summary(panel: pd.DataFrame) -> None:
    print(f"  Firms: {panel['company_name'].nunique():,} | "
          f"Treated: {panel.loc[panel['treated']==1,'company_name'].nunique():,} | "
          f"Control: {panel.loc[panel['treated']==0,'company_name'].nunique():,}")


## 4. Load & Clean Positions (Run Once)

In [ ]:
def clean_contained_positions(positions: pd.DataFrame) -> pd.DataFrame:
    FAR_FUTURE = pd.Timestamp("2099-12-31")
    df = positions.copy().reset_index(drop=True)
    df["_end_filled"] = df["enddate"].fillna(FAR_FUTURE)
    df["_row"] = df.index

    print("  Step 1/4: Self-joining …")
    keys = df[["_row", "user_id", "company_name", "startdate", "_end_filled", "seniority"]]
    pairs = keys.merge(
        keys.rename(columns={"_row": "_row_j", "startdate": "_start_j",
                              "_end_filled": "_end_j", "seniority": "_sen_j"}),
        on=["user_id", "company_name"],
    )

    print("  Step 2/4: Filtering …")
    contained = pairs[
        (pairs["_row"] != pairs["_row_j"])
        & (pairs["_start_j"] >= pairs["startdate"])
        & (pairs["_end_j"]   <= pairs["_end_filled"])
    ].copy()
    n_pairs = len(contained)
    print(f"  Found {n_pairs:,} containment pairs")

    if n_pairs == 0:
        print("  No contained positions — data unchanged.")
        return df.drop(columns=["_end_filled", "_row"])

    print("  Step 3/4: Classifying …")
    promotions = contained[contained["seniority"] <= contained["_sen_j"]]
    errors     = contained[contained["seniority"] >  contained["_sen_j"]]
    drop_rows  = set(errors["_row_j"].unique())

    shorten = pd.DataFrame(columns=["_row", "_new_end"])
    if len(promotions) > 0:
        shorten = (
            promotions.groupby("_row")["_start_j"].min().reset_index()
            .rename(columns={"_start_j": "_new_end_raw"})
        )
        shorten["_new_end"] = shorten["_new_end_raw"] - pd.Timedelta(days=1)
        shorten = shorten[~shorten["_row"].isin(drop_rows)]

    print("  Step 4/4: Applying edits …")
    rows_to_shorten = []
    if len(shorten) > 0:
        shorten_idx  = shorten.set_index("_row")["_new_end"]
        current_ends = df.loc[shorten_idx.index, "enddate"]
        mask         = shorten_idx < current_ends.fillna(FAR_FUTURE)
        rows_to_shorten = shorten_idx[mask].index.tolist()
        df.loc[rows_to_shorten, "enddate"] = shorten_idx[mask].values

    df = df[~df["_row"].isin(drop_rows)].drop(columns=["_end_filled", "_row"])
    print(f"  Done — shortened: {len(rows_to_shorten):,} | "
          f"dropped: {len(drop_rows):,} | rows remaining: {len(df):,}")
    return df.reset_index(drop=True)


def load_and_clean_positions(positions_path: str,
                             recruitment_industries: list) -> pd.DataFrame:
    print("=" * 60)
    print("LOADING AND CLEANING POSITIONS")
    print("=" * 60)
    pos = pd.read_csv(positions_path)
    pos["startdate"]    = pd.to_datetime(pos["startdate"],    errors="coerce")
    pos["enddate"]      = pd.to_datetime(pos["enddate"],      errors="coerce")
    pos["company_name"] = pos["company_name"].astype(str).str.strip().str.lower()
    pos = pos.dropna(subset=["company_name", "startdate", "user_id"]).copy()
    print(f"Rows loaded: {len(pos):,}")

    for col in ("rics_k400", "seniority"):
        if col not in pos.columns:
            raise ValueError(f"Required column '{col}' not found.")

    pos["seniority"] = pd.to_numeric(pos["seniority"], errors="coerce")
    pos = pos.dropna(subset=["seniority"]).copy()
    pos["seniority"] = pos["seniority"].astype(int)

    bad_firms = pos.loc[pos["rics_k400"].isin(recruitment_industries), "company_name"].unique()
    pos = pos[~pos["company_name"].isin(bad_firms)].copy()
    print(f"After dropping {len(bad_firms):,} recruitment firms: {len(pos):,} rows")

    print("\nResolving contained positions …")
    pos = clean_contained_positions(pos)
    print(f"\nClean positions ready: {len(pos):,} rows")
    return pos.reset_index(drop=True)


## 5. Treated Firms & Active Firm Filter

In [ ]:
def load_treated_firms(post1_path: str, post2_path: str,
                       treated_start: str, treated_end: str) -> set:
    postings = pd.concat([pd.read_csv(post1_path), pd.read_csv(post2_path)],
                         ignore_index=True)
    postings["post_date"] = pd.to_datetime(postings["post_date"], errors="coerce")
    postings["company"]   = postings["company"].astype(str).str.strip().str.lower()
    postings["is_integrator_gpt"] = (
        pd.to_numeric(postings["is_integrator_gpt"], errors="coerce").fillna(0).astype(int)
    )
    treated = set(
        postings[
            (postings["is_integrator_gpt"] == 1)
            & (postings["post_date"] >= treated_start)
            & (postings["post_date"] <  treated_end)
        ]["company"].unique()
    )
    print(f"Ever-treated firms: {len(treated):,}")
    return treated


def get_active_firms(positions: pd.DataFrame, hire_window_start: str,
                     hire_window_end: str, threshold: int) -> set:
    hs, he = pd.to_datetime(hire_window_start), pd.to_datetime(hire_window_end)
    counts = (
        positions[(positions["startdate"] >= hs) & (positions["startdate"] < he)]
        .groupby("company_name")["user_id"].nunique()
        .reset_index(name="total_hires")
    )
    return set(counts.loc[counts["total_hires"] >= threshold, "company_name"])


### Run: load once

In [ ]:
clean_positions = load_and_clean_positions(POSITIONS_PATH, RECRUITMENT_INDUSTRIES)
treated_firms   = load_treated_firms(POST1_PATH, POST2_PATH, TREATED_START, TREATED_END)
active_firms    = get_active_firms(clean_positions, HIRE_START, HIRE_END, THRESHOLD)
print(f"Active firms (>= {THRESHOLD} hires): {len(active_firms):,}")


## 6. Attach AI Exposure (for Stock Panel Enrichment)

In [ ]:
def attach_occ_exposure(positions: pd.DataFrame,
                        occ_exposure_path: str,
                        positions_onet_col: str = "onet_code",
                        exposure_onet_col: str  = EXPOSURE_ONET_COL,
                        exposure_value_col: str = EXPOSURE_VALUE_COL) -> pd.DataFrame:
    """Merge occupation-level AI exposure score and add high_exposure (median split)."""
    occ = pd.read_csv(occ_exposure_path)
    occ[exposure_onet_col]  = occ[exposure_onet_col].astype(str).str.strip()
    occ[exposure_value_col] = pd.to_numeric(occ[exposure_value_col], errors="coerce")

    pos = positions.copy()
    pos[positions_onet_col] = pos[positions_onet_col].astype(str).str.strip()

    merged = pos.merge(
        occ[[exposure_onet_col, exposure_value_col]]
           .drop_duplicates(subset=[exposure_onet_col]),
        left_on=positions_onet_col, right_on=exposure_onet_col, how="left",
    )
    med = merged[exposure_value_col].median(skipna=True)
    merged["high_exposure"] = np.where(
        merged[exposure_value_col].notna(),
        (merged[exposure_value_col] > med).astype(int),
        np.nan,
    )
    n_missing = merged["high_exposure"].isna().sum()
    merged = merged.dropna(subset=["high_exposure"]).copy()
    merged["high_exposure"] = merged["high_exposure"].astype(int)
    print(f"Exposure column: {exposure_value_col} | Median: {med:.4f} | "
          f"Dropped missing: {n_missing:,} | Remaining: {len(merged):,}")
    return merged, med


clean_positions_exposure, exposure_median = attach_occ_exposure(
    positions=clean_positions,
    occ_exposure_path=OCC_EXPOSURE_PATH,
)


## 7. Stock Panel (Flat — ge20)

Columns used by `analysis.R`:
`log_employees_total`, `employees_total`, `employees_highexp`,
`employees_lvl1`, `employees_lvl2`, `employees_lvl1_2`, `employees_lvl3_7`

In [ ]:
def build_stock_panel_enriched(positions, positions_exposure,
                               treated_firms, active_firms,
                               panel_start_q, panel_end_q, treat_q,
                               threshold, out_path):
    """
    Builds the stock panel with seniority and exposure sub-counts merged in.
    All columns needed by analysis.R are produced in a single file.
    """
    quarters = pd.period_range(panel_start_q, panel_end_q, freq="Q")

    pos     = positions[positions["company_name"].isin(active_firms)].copy()
    pos_exp = positions_exposure[positions_exposure["company_name"].isin(active_firms)].copy()

    # ── seniority indicators ──────────────────────────────────────────────────
    pos["is_lvl1"]  = (pos["seniority"] == 1).astype(int)
    pos["is_lvl2"]  = (pos["seniority"] == 2).astype(int)
    pos["is_lvl3_7"]= pos["seniority"].between(3, 7).astype(int)

    print(f"\nBuilding STOCK panel (threshold >= {threshold}) …")
    rows = []
    for q in tqdm(quarters, desc="Stock"):
        q_s, q_e = q.start_time, q.end_time
        active = pos[(pos["startdate"] <= q_e)
                     & (pos["enddate"].isna() | (pos["enddate"] >= q_s))]
        active_exp = pos_exp[(pos_exp["startdate"] <= q_e)
                             & (pos_exp["enddate"].isna() | (pos_exp["enddate"] >= q_s))]

        total  = active.groupby("company_name")["user_id"].nunique().rename("employees_total")
        lvl1   = active[active["is_lvl1"]  ==1].groupby("company_name")["user_id"].nunique().rename("employees_lvl1")
        lvl2   = active[active["is_lvl2"]  ==1].groupby("company_name")["user_id"].nunique().rename("employees_lvl2")
        lvl3_7 = active[active["is_lvl3_7"]==1].groupby("company_name")["user_id"].nunique().rename("employees_lvl3_7")
        highexp= (active_exp[active_exp["high_exposure"]==1]
                  .groupby("company_name")["user_id"].nunique()
                  .rename("employees_highexp"))

        qdf = (pd.DataFrame(total)
               .join(lvl1,    how="left")
               .join(lvl2,    how="left")
               .join(lvl3_7,  how="left")
               .join(highexp, how="left")
               .fillna(0).astype(int)
               .reset_index()
               .assign(quarter=q))
        rows.append(qdf)

    panel = make_firm_quarter_grid(active_firms, quarters).merge(
        pd.concat(rows, ignore_index=True), on=["company_name", "quarter"], how="left"
    )
    for c in ["employees_total", "employees_lvl1", "employees_lvl2",
              "employees_lvl3_7", "employees_highexp"]:
        panel[c] = panel[c].fillna(0).astype(int)

    panel["employees_lvl1_2"] = panel["employees_lvl1"] + panel["employees_lvl2"]
    panel = add_common_panel_columns(panel, treated_firms, treat_q)
    panel["log_employees_total"] = np.log1p(panel["employees_total"])
    print_firm_summary(panel)

    out_cols = ["firm_id", "company_name", "tq", "treated",
                "employees_total", "log_employees_total",
                "employees_lvl1", "employees_lvl2", "employees_lvl1_2",
                "employees_lvl3_7", "employees_highexp"]
    save_dta(panel[out_cols], out_path,
             int_cols=["firm_id", "tq", "treated"],
             float_cols=[c for c in out_cols
                         if c not in ("firm_id", "company_name", "tq", "treated")])
    return panel


stock_panel = build_stock_panel_enriched(
    clean_positions, clean_positions_exposure,
    treated_firms, active_firms,
    PANEL_START, PANEL_END, TREAT_Q, THRESHOLD,
    f"{OUT_DIR}firm_quarter_stock_active_ge{THRESHOLD}_chatgpt_treated_pretrend2021.dta",
)


## 8. New Hires Panel (Flat — ge20)

Columns used by `analysis.R`:
`log_new_hires_total`, `new_hires_total`, `new_hires_lvl1`, `new_hires_lvl2`, `new_hires_lvl3_7`

In [ ]:
def build_hires_panel_enriched(positions, treated_firms, active_firms,
                               panel_start_q, panel_end_q, treat_q,
                               threshold, out_path):
    """Flow panel with seniority sub-counts merged in."""
    quarters = pd.period_range(panel_start_q, panel_end_q, freq="Q")
    pos = positions[positions["company_name"].isin(active_firms)].copy()
    pos["is_lvl1"]   = (pos["seniority"] == 1).astype(int)
    pos["is_lvl2"]   = (pos["seniority"] == 2).astype(int)
    pos["is_lvl3_7"] = pos["seniority"].between(3, 7).astype(int)

    print(f"\nBuilding HIRES panel (threshold >= {threshold}) …")
    rows = []
    for q in tqdm(quarters, desc="Hires"):
        q_s = q.start_time;  q_e = (q + 1).start_time
        started = pos[(pos["startdate"] >= q_s) & (pos["startdate"] < q_e)]
        total  = started.groupby("company_name")["user_id"].nunique().rename("new_hires_total")
        lvl1   = started[started["is_lvl1"]  ==1].groupby("company_name")["user_id"].nunique().rename("new_hires_lvl1")
        lvl2   = started[started["is_lvl2"]  ==1].groupby("company_name")["user_id"].nunique().rename("new_hires_lvl2")
        lvl3_7 = started[started["is_lvl3_7"]==1].groupby("company_name")["user_id"].nunique().rename("new_hires_lvl3_7")
        qdf = (pd.DataFrame(total)
               .join(lvl1,   how="left")
               .join(lvl2,   how="left")
               .join(lvl3_7, how="left")
               .fillna(0).astype(int)
               .reset_index()
               .assign(quarter=q))
        rows.append(qdf)

    panel = make_firm_quarter_grid(active_firms, quarters).merge(
        pd.concat(rows, ignore_index=True), on=["company_name", "quarter"], how="left"
    )
    for c in ["new_hires_total", "new_hires_lvl1", "new_hires_lvl2", "new_hires_lvl3_7"]:
        panel[c] = panel[c].fillna(0).astype(int)
    panel = add_common_panel_columns(panel, treated_firms, treat_q)
    panel["log_new_hires_total"]   = np.log1p(panel["new_hires_total"])
    panel["log_new_hires_lvl1"]    = np.log1p(panel["new_hires_lvl1"])
    panel["log_new_hires_lvl2"]    = np.log1p(panel["new_hires_lvl2"])
    panel["log_new_hires_lvl3_7"]  = np.log1p(panel["new_hires_lvl3_7"])
    print_firm_summary(panel)

    out_cols = ["firm_id", "company_name", "tq", "treated", "post",
                "new_hires_total",  "log_new_hires_total",
                "new_hires_lvl1",   "log_new_hires_lvl1",
                "new_hires_lvl2",   "log_new_hires_lvl2",
                "new_hires_lvl3_7", "log_new_hires_lvl3_7"]
    save_dta(panel[out_cols], out_path,
             int_cols=["firm_id", "tq", "treated", "post"],
             float_cols=[c for c in out_cols
                         if c not in ("firm_id", "company_name", "tq", "treated", "post")])
    return panel


hires_panel = build_hires_panel_enriched(
    clean_positions, treated_firms, active_firms,
    PANEL_START, PANEL_END, TREAT_Q, THRESHOLD,
    f"{OUT_DIR}firm_quarter_new_hires_active_ge{THRESHOLD}_chatgpt_treated_pretrend2021.dta",
)


## 9. Separations Panel (Flat — ge20)

In [ ]:
def annotate_next_employer(positions: pd.DataFrame) -> pd.DataFrame:
    """Run on FULL cleaned positions so cross-firm moves are caught correctly."""
    df = positions.sort_values(
        ["user_id", "startdate", "enddate", "company_name"], kind="mergesort"
    ).copy()
    df["next_company_name"] = df.groupby("user_id")["company_name"].shift(-1)
    return df


def build_separations_panel(positions, treated_firms, active_firms,
                            panel_start_q, panel_end_q, treat_q,
                            threshold, out_path):
    if "next_company_name" not in positions.columns:
        raise ValueError("Run annotate_next_employer() first.")
    quarters = pd.period_range(panel_start_q, panel_end_q, freq="Q")
    pos = positions[positions["company_name"].isin(active_firms)].copy()

    print(f"\nBuilding SEPARATIONS panel (threshold >= {threshold}) …")
    rows = []
    for q in tqdm(quarters, desc="Separations"):
        ended = pos[
            pos["enddate"].notna()
            & (pos["enddate"] >= q.start_time)
            & (pos["enddate"] < (q + 1).start_time)
        ]
        exits = ended[
            ended["next_company_name"].isna()
            | (ended["next_company_name"] != ended["company_name"])
        ]
        s = exits.groupby("company_name")["user_id"].nunique().rename("separations_total")
        rows.append(s.reset_index().assign(quarter=q))

    panel = make_firm_quarter_grid(active_firms, quarters).merge(
        pd.concat(rows, ignore_index=True), on=["company_name", "quarter"], how="left"
    )
    panel["separations_total"] = panel["separations_total"].fillna(0).astype(int)
    panel = add_common_panel_columns(panel, treated_firms, treat_q)
    panel["log_separations_total"] = np.log1p(panel["separations_total"])
    print_firm_summary(panel)

    out_cols = ["firm_id", "company_name", "tq", "treated", "post",
                "separations_total", "log_separations_total"]
    save_dta(panel[out_cols], out_path,
             int_cols=["firm_id", "tq", "treated", "post"],
             float_cols=["separations_total", "log_separations_total"])
    return panel


clean_positions_with_next = annotate_next_employer(clean_positions)

seps_panel = build_separations_panel(
    clean_positions_with_next, treated_firms, active_firms,
    PANEL_START, PANEL_END, TREAT_Q, THRESHOLD,
    f"{OUT_DIR}firm_quarter_separations_active_ge{THRESHOLD}_chatgpt_treated_pretrend2021.dta",
)


## 10. Promotions Panel (Flat — ge20)

In [ ]:
def annotate_next_spell_for_promotions(positions: pd.DataFrame) -> pd.DataFrame:
    """Run on FULL cleaned positions before restricting to active firms."""
    df = positions.sort_values(
        ["user_id", "startdate", "enddate", "company_name"], kind="mergesort"
    ).copy()
    df["next_company_name"] = df.groupby("user_id")["company_name"].shift(-1)
    df["next_startdate"]    = df.groupby("user_id")["startdate"].shift(-1)
    df["next_seniority"]    = df.groupby("user_id")["seniority"].shift(-1)
    return df


def build_promotions_panel(positions, treated_firms, active_firms,
                           panel_start_q, panel_end_q, treat_q,
                           threshold, out_path):
    required = {"next_company_name", "next_startdate", "next_seniority"}
    if not required.issubset(positions.columns):
        raise ValueError("Run annotate_next_spell_for_promotions() first.")

    quarters = pd.period_range(panel_start_q, panel_end_q, freq="Q")
    pos = positions[positions["company_name"].isin(active_firms)].copy()
    pos = pos[pos["seniority"].between(1, 7)].copy()
    pos["is_promotable"] = pos["seniority"].between(1, 6).astype(int)

    print(f"\nBuilding PROMOTIONS panel (threshold >= {threshold}) …")
    rows = []
    for q in tqdm(quarters, desc="Promotions"):
        q_s, q_e = q.start_time, q.end_time

        active = pos[(pos["startdate"] <= q_e)
                     & (pos["enddate"].isna() | (pos["enddate"] >= q_s))]
        prom_stock = (active[active["is_promotable"]==1]
                      .groupby("company_name")["user_id"].nunique()
                      .rename("promotable_stock"))

        df_q    = pos[pos["seniority"].between(1,7) & pos["next_seniority"].between(1,7)].copy()
        promoted = df_q[
            (df_q["next_company_name"] == df_q["company_name"])
            & (df_q["next_seniority"] > df_q["seniority"])
            & df_q["next_startdate"].notna()
            & (df_q["next_startdate"] >= q_s)
            & (df_q["next_startdate"] < (q+1).start_time)
        ]
        promo_ct = (promoted.groupby("company_name")["user_id"].nunique()
                    .rename("promotions_total"))

        qdf = (pd.DataFrame(index=sorted(active_firms))
               .join(prom_stock, how="left")
               .join(promo_ct,   how="left")
               .fillna(0).astype(int)
               .reset_index().rename(columns={"index": "company_name"})
               .assign(quarter=q))
        rows.append(qdf)

    panel = pd.concat(rows, ignore_index=True)
    panel = add_common_panel_columns(panel, treated_firms, treat_q)
    panel = panel.sort_values(["company_name", "quarter"])
    panel["lag_promotable_stock"] = panel.groupby("company_name")["promotable_stock"].shift(1)
    panel["promo_rate"]           = np.where(
        panel["lag_promotable_stock"] > 0,
        panel["promotions_total"] / panel["lag_promotable_stock"], np.nan
    )
    panel["log_promotions_total"] = np.log1p(panel["promotions_total"])
    panel["log_promo_rate"]       = np.log(panel["promo_rate"] + 1e-6)
    print_firm_summary(panel)

    out_cols = ["firm_id", "company_name", "tq", "treated", "post",
                "promotions_total", "log_promotions_total",
                "promotable_stock", "lag_promotable_stock",
                "promo_rate",       "log_promo_rate"]
    save_dta(panel[out_cols], out_path,
             int_cols=["firm_id", "tq", "treated", "post"],
             float_cols=[c for c in out_cols
                         if c not in ("firm_id","company_name","tq","treated","post")])
    return panel


clean_positions_promos = annotate_next_spell_for_promotions(clean_positions)

promos_panel = build_promotions_panel(
    clean_positions_promos, treated_firms, active_firms,
    PANEL_START, PANEL_END, TREAT_Q, THRESHOLD,
    f"{OUT_DIR}firm_quarter_promotions_active_ge{THRESHOLD}_chatgpt_treated_pretrend2021.dta",
)


## 11. Staggered Panels (for CS + SA DiD)

These add `first_treat_tq` (first quarter with a GenAI integrator posting) to each
of the four flat panels. Never-treated firms get `first_treat_tq = NaN`.

In [ ]:
def load_first_treat_tq(post1_path: str, post2_path: str,
                        treated_start: str, treated_end: str) -> pd.DataFrame:
    """
    Returns a DataFrame (company_name, first_treat_tq) where first_treat_tq
    is the Stata quarter integer of the firm's first GenAI integrator posting.
    """
    postings = pd.concat([pd.read_csv(post1_path), pd.read_csv(post2_path)],
                         ignore_index=True)
    postings["post_date"] = pd.to_datetime(postings["post_date"], errors="coerce")
    postings["company"]   = postings["company"].astype(str).str.strip().str.lower()
    postings["is_integrator_gpt"] = (
        pd.to_numeric(postings["is_integrator_gpt"], errors="coerce").fillna(0).astype(int)
    )
    integrators = postings[
        (postings["is_integrator_gpt"] == 1)
        & (postings["post_date"] >= treated_start)
        & (postings["post_date"] <  treated_end)
    ]
    first_dates = (
        integrators.groupby("company")["post_date"].min().reset_index()
        .rename(columns={"company": "company_name", "post_date": "first_treat_date"})
    )
    first_dates["first_treat_period"] = pd.PeriodIndex(
        first_dates["first_treat_date"], freq="Q"
    )
    first_dates["first_treat_tq"] = first_dates["first_treat_period"].apply(
        lambda p: (p - STATA_EPOCH).n
    ).astype(int)
    return first_dates[["company_name", "first_treat_tq"]]


def make_staggered(flat_panel: pd.DataFrame,
                   first_treat_df: pd.DataFrame,
                   out_path: str,
                   value_cols: list) -> pd.DataFrame:
    """Merge first_treat_tq onto a flat panel and save as staggered .dta."""
    panel = flat_panel.merge(first_treat_df, on="company_name", how="left")
    # never-treated firms: first_treat_tq stays NaN (did + sunab handle 0 / NA)
    int_cols   = ["firm_id", "tq", "treated", "post"]
    float_cols = value_cols + ["first_treat_tq"]
    # first_treat_tq: keep as float so NaN is preserved (R reads it fine)
    out = panel[["firm_id", "company_name", "tq", "treated", "post",
                 "first_treat_tq"] + value_cols].copy()
    for c in int_cols:
        out[c] = out[c].astype(int)
    for c in value_cols:
        out[c] = out[c].astype(float)
    # first_treat_tq: float (NaN for never-treated)
    pyreadstat.write_dta(out, out_path, version=14)
    print(f"  Saved → {out_path}  ({len(out):,} rows)")
    return panel


first_treat_df = load_first_treat_tq(POST1_PATH, POST2_PATH, TREATED_START, TREATED_END)
print(f"Firms with a first_treat_tq: {first_treat_df['company_name'].nunique():,}")


In [ ]:
make_staggered(
    stock_panel, first_treat_df,
    f"{OUT_DIR}stock_staggered.dta",
    value_cols=["employees_total", "log_employees_total"],
)

make_staggered(
    hires_panel, first_treat_df,
    f"{OUT_DIR}hires_staggered.dta",
    value_cols=["new_hires_total", "log_new_hires_total"],
)

make_staggered(
    seps_panel, first_treat_df,
    f"{OUT_DIR}seps_staggered.dta",
    value_cols=["separations_total", "log_separations_total"],
)

make_staggered(
    promos_panel, first_treat_df,
    f"{OUT_DIR}promos_staggered.dta",
    value_cols=["promotions_total", "log_promotions_total"],
)

print("\nAll 8 .dta files saved.")
